# Lesson 1: The basics
Run each cell with **Shift+Enter**, top to bottom. Notes are in `langchain.md`.

## Step 0: Load your API key
This reads `GROQ_API_KEY` from your `.env` file.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print('Key found:', bool(os.getenv('GROQ_API_KEY')) and os.getenv('GROQ_API_KEY') != 'paste-your-key-here')

## Step 1: Call a model
The model alone can only talk. It cannot do anything in the world.

In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model('groq:openai/gpt-oss-120b')
reply = model.invoke('What is a stock, in one sentence?')
print(reply.content)

A stock is a share of ownership in a corporation that entitles the holder to a proportional claim on the company’s assets, earnings, and voting rights.


## Step 2: Messages
A conversation is a list of messages, each with a role.

In [2]:
messages = [
    {'role': 'system', 'content': 'You explain money topics to a beginner. Be short.'},
    {'role': 'user', 'content': 'What does P/E ratio mean?'},
]
print(model.invoke(messages).content)

**P/E ratio = Price‑to‑Earnings ratio**

- **Price** – the current market price of one share of a company’s stock.  
- **Earnings** – the company’s net profit per share (usually over the last 12 months).

**Formula:**  

\[
\text{P/E} = \frac{\text{Share price}}{\text{Earnings per share (EPS)}}
\]

**What it tells you:**  
- A **high P/E** suggests investors expect strong future growth (they’re willing to pay more for each dollar of earnings).  
- A **low P/E** can mean the stock is cheap, or that the market expects slower growth or higher risk.

**Quick example:**  
If a stock trades at \$50 and its EPS is \$5, the P/E = 50 ÷ 5 = **10**. That means you’re paying \$10 for every \$1 of earnings.


## Step 3: Make a tool
A tool is a normal Python function. The docstring tells the model when to use it.

In [3]:
from langchain.tools import tool

@tool
def get_price(ticker: str) -> str:
    """Get the latest price of a stock. Input is a ticker symbol like AAPL."""
    return f'{ticker} costs $100'  # fake data for now

print(get_price.invoke({'ticker': 'AAPL'}))

AAPL costs $100


## Step 4: Make an agent
The agent loops: model thinks, asks for a tool, tool runs, model reads the result, repeat until it answers.

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model='groq:openai/gpt-oss-120b',
    tools=[get_price],
    system_prompt='You are a careful stock research assistant.',
)

result = agent.invoke({'messages': [{'role': 'user', 'content': 'How much is AAPL?'}]})
print(result['messages'][-1].content)

Apple Inc. (AAPL) is currently priced at **$100** per share.


## Step 5: Watch the loop
Print every message to see the tool request and the tool result in the middle.

In [5]:
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

How much is AAPL?
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_809724e4-26e0-4399-a029-a4163208fab5)
 Call ID: fc_809724e4-26e0-4399-a029-a4163208fab5
  Args:
    ticker: AAPL
================================= Tool Message =================================
Name: get_price

AAPL costs $100
================================== Ai Message ==================================

Apple Inc. (AAPL) is currently priced at **$100** per share.


## Exercise
Add a second tool, `get_company_name(ticker)`. Add it to `tools=[...]` and ask: *"What is AAPL and how much is it?"*

In [ ]:
# your code here